# Vault Auto-Unseal with AWS KMS — Web Identity/OIDC and Terraform

This notebook is equivalent to `Auto_unseal_AWS_WebIdentity_OIDC.ipynb`, but Terraform manages every AWS resource: the IAM OIDC provider, federated role, least-privilege role policy, KMS key, alias, and key policy. Kubernetes, Minikube, ngrok, Helm, and Vault remain procedural demo steps.

```text
Kubernetes ServiceAccount JWT → ngrok OIDC issuer → AWS STS
  → Terraform-managed IAM role → Terraform-managed KMS key → Vault auto-unseal
```

Web Identity parameters for the AWS KMS seal are supported from Vault 1.8.4. This notebook uses Vault Enterprise 2.1.0. The projected token requests a one-hour lifetime and the kubelet rotates it before expiry. Keep ngrok and `kubectl proxy` running whenever STS may need to validate a token or a Vault pod starts.

The final cleanup is intentionally ordered: local Vault/Minikube/Podman resources first, then `terraform destroy`. AWS KMS deletion is asynchronous; Terraform removes the key from state after AWS schedules its deletion using the seven-day recovery window.

## 1. Start OIDC-enabled Minikube and ngrok

Before running, configure ngrok and export a stable reserved domain: `NGROK_DOMAIN=your-domain.ngrok.app`. This lab exposes an authenticated `kubectl proxy`; stop the tunnel immediately after the demo.

Start Minikube with `--driver=podman` and the `workshop-oidc-tf` profile. Using another profile or letting Minikube pick `vfkit` is what produced the DNS failures.

In [ ]:
import csv
import json
import os
import pathlib
import re
import shutil
import subprocess
import time
import urllib.request

NGROK_DOMAIN = os.environ.get('NGROK_DOMAIN', '').strip().removeprefix('https://').rstrip('/')
if not NGROK_DOMAIN:
    raise ValueError('Export NGROK_DOMAIN=your-stable-domain.ngrok.app')
for binary in ('aws', 'ngrok', 'kubectl', 'minikube', 'podman', 'terraform', 'helm'):
    if not shutil.which(binary):
        raise RuntimeError(f'Required executable not found: {binary}')

MINIKUBE_PROFILE = 'workshop-oidc-tf'
NAMESPACE = 'vault'
SERVICE_ACCOUNT = 'vault'
OIDC_ISSUER_URL = f'https://{NGROK_DOMAIN}'
REGION = os.environ.get('AWS_REGION', 'eu-west-3')
TF_WORKDIR = pathlib.Path('/tmp/vault-oidc-terraform')
ARTIFACT_WORKDIR = pathlib.Path('/tmp/vault-oidc-tf-artifacts')
os.environ.update({
    'AWS_REGION': REGION,
    'OIDC_ISSUER_URL': OIDC_ISSUER_URL,
    'VAULT_K8S_NAMESPACE': NAMESPACE,
    'VAULT_SERVICE_ACCOUNT': SERVICE_ACCOUNT,
})

subprocess.run([
    'minikube', 'start', '-p', MINIKUBE_PROFILE, '--driver', 'podman',
    f'--extra-config=apiserver.service-account-issuer={OIDC_ISSUER_URL}',
    f'--extra-config=apiserver.service-account-jwks-uri={OIDC_ISSUER_URL}/openid/v1/jwks',
], check=True)
subprocess.run(['kubectl', 'config', 'use-context', MINIKUBE_PROFILE], check=True)

# A previous/interrupted kernel may have left the reserved ngrok endpoint
# online. Clean only this demo domain and proxy port before taking ownership.
for process_name in ('ngrok_process', 'kubectl_proxy_process'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
subprocess.run(['pkill', '-f', rf'ngrok.*{re.escape(NGROK_DOMAIN)}'], check=False)
subprocess.run(['pkill', '-f', r'kubectl.*proxy.*--port=8002'], check=False)
time.sleep(2)

kubectl_proxy_process = subprocess.Popen([
    'kubectl', '--context', MINIKUBE_PROFILE, 'proxy',
    '--address=127.0.0.1', '--port=8002', '--accept-hosts=.*',
], stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
time.sleep(2)
ngrok_process = subprocess.Popen(
    ['ngrok', 'http', '8002', f'--url={OIDC_ISSUER_URL}'],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True,
)

discovery_url = f'{OIDC_ISSUER_URL}/.well-known/openid-configuration'
for attempt in range(1, 31):
    if kubectl_proxy_process.poll() is not None:
        raise RuntimeError(f'kubectl proxy stopped: {kubectl_proxy_process.stderr.read()}')
    if ngrok_process.poll() is not None:
        raise RuntimeError(f'ngrok stopped: {ngrok_process.stderr.read()}')
    try:
        with urllib.request.urlopen(discovery_url, timeout=5) as response:
            discovery = json.load(response)
        break
    except Exception:
        if attempt == 30:
            raise
        time.sleep(2)
assert discovery['issuer'].rstrip('/') == OIDC_ISSUER_URL
with urllib.request.urlopen(discovery['jwks_uri'], timeout=10) as response:
    jwks = json.load(response)
assert jwks.get('keys')

tls_chain = subprocess.check_output(
    ['openssl', 's_client', '-showcerts', '-servername', NGROK_DOMAIN, '-connect', f'{NGROK_DOMAIN}:443'],
    input=b'', stderr=subprocess.DEVNULL,
)
certificates = re.findall(b'-----BEGIN CERTIFICATE-----.*?-----END CERTIFICATE-----', tls_chain, re.DOTALL)
fingerprint_line = subprocess.check_output(
    ['openssl', 'x509', '-noout', '-fingerprint', '-sha1'], input=certificates[-1],
).decode().strip()
OIDC_THUMBPRINT = fingerprint_line.split('=', 1)[1].replace(':', '').lower()
print(f'✓ Public issuer: {OIDC_ISSUER_URL}')
print(f'✓ Published JWKS keys: {len(jwks["keys"])}')

## 2. Load AWS provisioning credentials

These credentials are consumed by the Terraform AWS provider only. They are not stored in Terraform configuration or mounted into Kubernetes.

In [ ]:
with open('vault_test_accessKeys.csv', encoding='utf-8-sig') as csvfile:
    provisioning_credentials = next(csv.DictReader(csvfile))
os.environ['AWS_ACCESS_KEY_ID'] = provisioning_credentials['Access key ID'].strip()
os.environ['AWS_SECRET_ACCESS_KEY'] = provisioning_credentials['Secret access key'].strip()
os.environ.pop('AWS_SESSION_TOKEN', None)
os.environ.pop('AWS_SECURITY_TOKEN', None)
identity = subprocess.check_output([
    'aws', 'sts', 'get-caller-identity', '--query', 'Arn', '--output', 'text',
], text=True).strip()
print(f'✓ Terraform provisioning identity: {identity}')

## 3. Generate the Terraform configuration

The files follow HashiCorp conventions: version constraints and provider configuration are separated from variables, resources, and outputs. No credential is written to disk or state.

In [ ]:
%%bash
set -euo pipefail
TF_WORKDIR=/tmp/vault-oidc-terraform
mkdir -p "${TF_WORKDIR}"
cat > "${TF_WORKDIR}/terraform.tf" <<'EOF'
terraform {
  required_version = ">= 1.14"

  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 6.0"
    }
  }
}
EOF
cat > "${TF_WORKDIR}/providers.tf" <<'EOF'
provider "aws" {
  region = var.aws_region

  default_tags {
    tags = {
      ManagedBy = "Terraform"
      Project   = "vault-auto-unseal-oidc"
    }
  }
}
EOF
cat > "${TF_WORKDIR}/variables.tf" <<'EOF'
variable "aws_region" {
  description = "AWS region containing the Vault auto-unseal KMS key."
  type        = string
}

variable "kms_alias" {
  description = "Alias assigned to the Vault auto-unseal KMS key."
  type        = string
  default     = "alias/vault-auto-unseal-oidc-tf"
}

variable "oidc_issuer_url" {
  description = "Public HTTPS Kubernetes ServiceAccount issuer URL."
  type        = string

  validation {
    condition     = startswith(var.oidc_issuer_url, "https://")
    error_message = "oidc_issuer_url must use HTTPS."
  }
}

variable "oidc_thumbprint" {
  description = "SHA-1 thumbprint of the OIDC provider TLS certificate chain."
  type        = string
  sensitive   = true

  validation {
    condition     = can(regex("^[0-9a-fA-F]{40}$", var.oidc_thumbprint))
    error_message = "oidc_thumbprint must contain 40 hexadecimal characters."
  }
}

variable "role_name" {
  description = "Name of the IAM role assumed by Vault through Web Identity."
  type        = string
  default     = "vault-kms-unseal-oidc-tf"
}

variable "service_account_name" {
  description = "Kubernetes ServiceAccount name used by Vault."
  type        = string
  default     = "vault"
}

variable "service_account_namespace" {
  description = "Kubernetes namespace containing the Vault ServiceAccount."
  type        = string
  default     = "vault"
}
EOF
cat > "${TF_WORKDIR}/main.tf" <<'EOF'
data "aws_caller_identity" "current" {}

locals {
  oidc_hostpath          = trimprefix(var.oidc_issuer_url, "https://")
  service_account_subject = "system:serviceaccount:${var.service_account_namespace}:${var.service_account_name}"
}

resource "aws_iam_openid_connect_provider" "main" {
  url             = var.oidc_issuer_url
  client_id_list  = ["sts.amazonaws.com"]
  thumbprint_list = [var.oidc_thumbprint]
}

data "aws_iam_policy_document" "assume_role" {
  statement {
    effect  = "Allow"
    actions = ["sts:AssumeRoleWithWebIdentity"]

    principals {
      type        = "Federated"
      identifiers = [aws_iam_openid_connect_provider.main.arn]
    }

    condition {
      test     = "StringEquals"
      variable = "${local.oidc_hostpath}:aud"
      values   = ["sts.amazonaws.com"]
    }

    condition {
      test     = "StringEquals"
      variable = "${local.oidc_hostpath}:sub"
      values   = [local.service_account_subject]
    }
  }
}

resource "aws_iam_role" "vault" {
  name                 = var.role_name
  assume_role_policy   = data.aws_iam_policy_document.assume_role.json
  description          = "Web Identity role for Vault AWS KMS auto-unseal"
  max_session_duration = 3600
}

data "aws_iam_policy_document" "kms_key" {
  statement {
    sid     = "EnableRootAccountFullAccess"
    effect  = "Allow"
    actions = ["kms:*"]
    resources = ["*"]

    principals {
      type        = "AWS"
      identifiers = ["arn:aws:iam::${data.aws_caller_identity.current.account_id}:root"]
    }
  }

  statement {
    sid       = "AllowVaultWebIdentityRole"
    effect    = "Allow"
    actions   = ["kms:Decrypt", "kms:DescribeKey", "kms:Encrypt"]
    resources = ["*"]

    principals {
      type        = "AWS"
      identifiers = [aws_iam_role.vault.arn]
    }
  }
}

resource "aws_kms_key" "vault" {
  description             = "Vault auto-unseal key using Kubernetes Web Identity"
  deletion_window_in_days = 7
  enable_key_rotation     = true
  policy                  = data.aws_iam_policy_document.kms_key.json
}

resource "aws_kms_alias" "vault" {
  name          = var.kms_alias
  target_key_id = aws_kms_key.vault.key_id
}

data "aws_iam_policy_document" "kms_access" {
  statement {
    effect    = "Allow"
    actions   = ["kms:Decrypt", "kms:DescribeKey", "kms:Encrypt"]
    resources = [aws_kms_key.vault.arn]
  }
}

resource "aws_iam_role_policy" "kms_access" {
  name   = "vault-kms-auto-unseal"
  role   = aws_iam_role.vault.id
  policy = data.aws_iam_policy_document.kms_access.json
}
EOF
cat > "${TF_WORKDIR}/outputs.tf" <<'EOF'
output "kms_key_arn" {
  description = "ARN of the KMS key used for Vault auto-unseal."
  value       = aws_kms_key.vault.arn
}

output "kms_key_id" {
  description = "ID of the KMS key used for Vault auto-unseal."
  value       = aws_kms_key.vault.key_id
}

output "oidc_provider_arn" {
  description = "ARN of the IAM OIDC provider trusted by the Vault role."
  value       = aws_iam_openid_connect_provider.main.arn
}

output "role_arn" {
  description = "ARN of the IAM role assumed by Vault through Web Identity."
  value       = aws_iam_role.vault.arn
}
EOF
terraform -chdir="${TF_WORKDIR}" fmt
echo "✓ Terraform files written to ${TF_WORKDIR}"

## 4. Initialize, validate, and apply Terraform

In [ ]:
terraform_env = os.environ.copy()
terraform_env.update({
    'TF_IN_AUTOMATION': '1',
    'TF_VAR_aws_region': REGION,
    'TF_VAR_oidc_issuer_url': OIDC_ISSUER_URL,
    'TF_VAR_oidc_thumbprint': OIDC_THUMBPRINT,
    'TF_VAR_service_account_name': SERVICE_ACCOUNT,
    'TF_VAR_service_account_namespace': NAMESPACE,
})
for command in (
    ['terraform', f'-chdir={TF_WORKDIR}', 'init'],
    ['terraform', f'-chdir={TF_WORKDIR}', 'validate'],
    ['terraform', f'-chdir={TF_WORKDIR}', 'apply', '-auto-approve'],
):
    subprocess.run(command, env=terraform_env, check=True)
terraform_outputs = json.loads(subprocess.check_output(
    ['terraform', f'-chdir={TF_WORKDIR}', 'output', '-json'], env=terraform_env, text=True,
))
KMS_KEY_ID = terraform_outputs['kms_key_id']['value']
VAULT_AWS_ROLE_ARN = terraform_outputs['role_arn']['value']
os.environ['KMS_KEY_ID'] = KMS_KEY_ID
os.environ['VAULT_AWS_ROLE_ARN'] = VAULT_AWS_ROLE_ARN
print(f'✓ KMS key ID: {KMS_KEY_ID}')
print(f'✓ Vault role ARN: {VAULT_AWS_ROLE_ARN}')

## 5. Create the ServiceAccount, TLS, and license Secrets

In [ ]:
%%bash
set -euo pipefail
WORKDIR=/tmp/vault-oidc-tf-artifacts
mkdir -p "${WORKDIR}"
test -s vault.hclic || { echo 'vault.hclic not found' >&2; exit 1; }
kubectl create namespace "${VAULT_K8S_NAMESPACE}" --dry-run=client -o yaml | kubectl apply -f -
kubectl create serviceaccount "${VAULT_SERVICE_ACCOUNT}" -n "${VAULT_K8S_NAMESPACE}" \
  --dry-run=client -o yaml | kubectl apply -f -
openssl req -x509 -nodes -newkey rsa:2048 -days 30 \
  -keyout "${WORKDIR}/vault.key" -out "${WORKDIR}/vault.crt" \
  -subj '/CN=vault.vault.svc' \
  -addext 'subjectAltName=DNS:vault,DNS:vault.vault,DNS:vault.vault.svc,DNS:*.vault-internal,DNS:*.vault-internal.vault.svc,IP:127.0.0.1'
cp "${WORKDIR}/vault.crt" "${WORKDIR}/vault.ca"
kubectl create secret generic vault-ha-tls -n "${VAULT_K8S_NAMESPACE}" \
  --from-file=vault.key="${WORKDIR}/vault.key" \
  --from-file=vault.crt="${WORKDIR}/vault.crt" \
  --from-file=vault.ca="${WORKDIR}/vault.ca" \
  --dry-run=client -o yaml | kubectl apply -f -
kubectl create secret generic vault-ent-license -n "${VAULT_K8S_NAMESPACE}" \
  --from-file=license=vault.hclic --dry-run=client -o yaml | kubectl apply -f -
echo '✓ Kubernetes prerequisites ready'

## 6. Generate Helm values and deploy Vault

In [ ]:
%%bash
set -euo pipefail
WORKDIR=/tmp/vault-oidc-tf-artifacts
curl -fsS --max-time 15 "${OIDC_ISSUER_URL}/.well-known/openid-configuration" >/dev/null
curl -fsS --max-time 15 "${OIDC_ISSUER_URL}/openid/v1/jwks" >/dev/null
cat > "${WORKDIR}/overrides.yaml" <<EOF
global:
  enabled: true
  tlsDisable: false
injector:
  enabled: false
server:
  image:
    repository: docker.io/hashicorp/vault-enterprise
    tag: 2.1.0-ent
  enterpriseLicense:
    secretName: vault-ent-license
  serviceAccount:
    create: false
    name: ${VAULT_SERVICE_ACCOUNT}
  extraEnvironmentVars:
    VAULT_CACERT: /vault/userconfig/vault-ha-tls/vault.ca
    VAULT_TLSCERT: /vault/userconfig/vault-ha-tls/vault.crt
    VAULT_TLSKEY: /vault/userconfig/vault-ha-tls/vault.key
  volumes:
    - name: userconfig-vault-ha-tls
      secret:
        defaultMode: 420
        secretName: vault-ha-tls
    - name: aws-web-identity
      projected:
        defaultMode: 420
        sources:
          - serviceAccountToken:
              audience: sts.amazonaws.com
              expirationSeconds: 3600
              path: token
  volumeMounts:
    - name: userconfig-vault-ha-tls
      mountPath: /vault/userconfig/vault-ha-tls
      readOnly: true
    - name: aws-web-identity
      mountPath: /var/run/secrets/aws
      readOnly: true
  standalone:
    enabled: false
  affinity: ""
  ha:
    enabled: true
    replicas: 3
    raft:
      enabled: true
      setNodeId: true
      config: |
        ui = true
        listener "tcp" {
          address            = "[::]:8200"
          cluster_address    = "[::]:8201"
          tls_disable        = 0
          tls_cert_file      = "/vault/userconfig/vault-ha-tls/vault.crt"
          tls_key_file       = "/vault/userconfig/vault-ha-tls/vault.key"
          tls_client_ca_file = "/vault/userconfig/vault-ha-tls/vault.ca"
        }
        storage "raft" {
          path = "/vault/data"
          retry_join {
            leader_api_addr       = "https://vault-0.vault-internal:8200"
            leader_ca_cert_file   = "/vault/userconfig/vault-ha-tls/vault.ca"
            leader_tls_servername = "vault-0.vault-internal"
          }
        }
        seal "awskms" {
          region                  = "${AWS_REGION}"
          kms_key_id              = "${KMS_KEY_ID}"
          role_arn                = "${VAULT_AWS_ROLE_ARN}"
          role_session_name       = "vault-auto-unseal-oidc-tf"
          web_identity_token_file = "/var/run/secrets/aws/token"
        }
        disable_mlock = true
        service_registration "kubernetes" {}
EOF
helm repo add hashicorp https://helm.releases.hashicorp.com --force-update
helm upgrade --install vault hashicorp/vault -n "${VAULT_K8S_NAMESPACE}" \
  -f "${WORKDIR}/overrides.yaml" --force-conflicts
for attempt in $(seq 1 60); do
  kubectl get pod vault-0 -n "${VAULT_K8S_NAMESPACE}" >/dev/null 2>&1 && break
  sleep 2
done
kubectl wait pod/vault-0 -n "${VAULT_K8S_NAMESPACE}" \
  --for=jsonpath='{.status.phase}'=Running --timeout=180s
echo '✓ Vault pods created'

## 7. Initialize once and verify three-node auto-unseal

In [ ]:
%%bash
set -euo pipefail
WORKDIR=/tmp/vault-oidc-tf-artifacts
curl -fsS --max-time 15 "${OIDC_ISSUER_URL}/.well-known/openid-configuration" >/dev/null
curl -fsS --max-time 15 "${OIDC_ISSUER_URL}/openid/v1/jwks" >/dev/null
umask 077
status_json=$(kubectl exec -n "${VAULT_K8S_NAMESPACE}" vault-0 -- vault status -format=json 2>/dev/null || true)
if echo "${status_json}" | grep -q '"initialized": false'; then
  kubectl exec -n "${VAULT_K8S_NAMESPACE}" vault-0 -- \
    vault operator init -format=json -recovery-shares=1 -recovery-threshold=1 \
    > "${WORKDIR}/vault-init.json"
  chmod 600 "${WORKDIR}/vault-init.json"
  echo "✓ Initialization material saved to ${WORKDIR}/vault-init.json"
fi
kubectl delete pod vault-0 -n "${VAULT_K8S_NAMESPACE}"
for pod in vault-0 vault-1 vault-2; do
  for attempt in $(seq 1 90); do
    status=$(kubectl exec -n "${VAULT_K8S_NAMESPACE}" "${pod}" -- vault status -format=json 2>/dev/null || true)
    if echo "${status}" | grep -q '"initialized": true' && echo "${status}" | grep -q '"sealed": false'; then
      echo "✓ ${pod} initialized and unsealed"
      break
    fi
    sleep 2
  done
done

## 8. Complete local cleanup before AWS

This irreversibly removes Vault initialization material, the Helm release, namespace, Minikube profile, ngrok/proxy processes, Podman demo image, and local image archive. The Terraform working directory is retained until AWS destroy succeeds.

In [ ]:
for process_name in ('ngrok_process', 'kubectl_proxy_process'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
        process.wait(timeout=10)
subprocess.run(['pkill', '-f', rf'ngrok.*{re.escape(NGROK_DOMAIN)}'], check=False)
subprocess.run(['pkill', '-f', r'kubectl.*proxy.*--port=8002'], check=False)
print('✓ Stale ngrok and kubectl proxy processes removed')
subprocess.run(['helm', 'uninstall', 'vault', '-n', NAMESPACE], check=False)
subprocess.run(['kubectl', 'delete', 'namespace', NAMESPACE, '--ignore-not-found=true', '--wait=true', '--timeout=180s'], check=False)
subprocess.run(['minikube', 'delete', '-p', MINIKUBE_PROFILE], check=False)
subprocess.run(['podman', 'image', 'rm', '--force', 'docker.io/hashicorp/vault-enterprise:2.1.0-ent'], check=False)
pathlib.Path('/tmp/vault-enterprise-2.1.0-ent-tf.tar').unlink(missing_ok=True)
shutil.rmtree(ARTIFACT_WORKDIR, ignore_errors=True)
print('✓ Vault, namespace, Minikube, Podman image, tunnel, and local secrets removed')

## 9. Complete AWS cleanup with Terraform

Terraform destroys the role, inline policy, alias, KMS key, key policy, and IAM OIDC provider. AWS schedules KMS key deletion for seven days. After a successful destroy, the notebook removes Terraform state and provider cache from `/tmp`.

In [ ]:
subprocess.run([
    'terraform', f'-chdir={TF_WORKDIR}', 'destroy', '-auto-approve',
], env=terraform_env, check=True)
shutil.rmtree(TF_WORKDIR)
print('✓ AWS resources destroyed and Terraform state/cache removed')